# NB3 — Core-7 × FashionCLIP embedding validation

Notebook này kiểm tra xem **mọi item còn lại sau Core-7/DROP** có embedding FashionCLIP dùng được hay không.

Nó kiểm tra:

1. cache đúng model `patrickjohncyh/fashion-clip`;
2. tensor có dạng `[N, 512]` và số hàng khớp `item_ids`;
3. không có duplicate ID, NaN/Inf, zero vector hoặc norm sai;
4. item trong từng `category_clean_{split}.jsonl` có metadata tương ứng;
5. mọi item cần dùng đều có embedding hợp lệ.

Nếu cả ba split pass, **không tạo thêm JSONL giống hệt**: các file `category_clean_*` hiện tại được dùng luôn làm final clean positives và pipeline có thể chuyển sang negative sampling.

## 1. Runtime portable

Notebook không tự mount Drive hoặc tự checkout/pull Git. Hãy mở từ repository đã clone. Trên Colab, mount Drive và set các environment variable trước khi chạy nếu artifact nằm trên Drive.

In [ ]:
# Không có setup bắt buộc dành riêng cho Colab.
# Xem README: FASHION_PROJECT_ROOT, FASHION_ARTIFACT_ROOT,
# FASHION_EMBEDDING_CACHE.

## 2. Tìm repository và load path config

Mặc định local dùng `./data`. Có thể override bằng environment variable hoặc `configs/data_paths.local.json`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/ThinhTran2208/opisoverated.git"
AUTO_CLONE_REPO = False  # Chỉ bật khi runtime chưa có repository, ví dụ Colab mới.


def find_repo_root(start: Path = Path.cwd()):
    explicit = os.environ.get("FASHION_PROJECT_ROOT")
    if explicit:
        candidate = Path(explicit).expanduser().resolve()
        if (candidate / "src/data/prepare_core7_dataset.py").exists():
            return candidate
        raise FileNotFoundError(f"FASHION_PROJECT_ROOT không hợp lệ: {candidate}")

    current = start.expanduser().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src/data/prepare_core7_dataset.py").exists():
            return candidate
    return None


REPO_ROOT = find_repo_root()
if REPO_ROOT is None and AUTO_CLONE_REPO:
    REPO_ROOT = (Path.cwd() / "opisoverated").resolve()
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)

if REPO_ROOT is None:
    raise RuntimeError(
        "Không tìm thấy repository. Hãy mở notebook từ repo đã clone, "
        "set FASHION_PROJECT_ROOT, hoặc bật AUTO_CLONE_REPO=True."
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.runtime_paths import load_runtime_paths

RUNTIME_PATHS = load_runtime_paths(repo_root=REPO_ROOT)
print("Repo root      :", REPO_ROOT)
print("Path config    :", RUNTIME_PATHS.config_path)
print("Artifact root  :", RUNTIME_PATHS.artifact_root)

## 3. Xác định input

- `CORE7_DIR`: output full của NB2.
- `CACHE_PATH`: FashionCLIP cache.

Cả hai được lấy từ runtime path config, không phụ thuộc tên shortcut Drive.

In [ ]:
CORE7_DIR = RUNTIME_PATHS.core7_dir
CACHE_PATH = RUNTIME_PATHS.embedding_cache

if not CORE7_DIR.exists():
    raise FileNotFoundError(
        f"Không tìm thấy Core-7 output folder: {CORE7_DIR}. Hãy chạy NB2 full trước."
    )
if not CACHE_PATH.is_file():
    raise FileNotFoundError(
        f"Không tìm thấy FashionCLIP cache: {CACHE_PATH}. "
        "Hãy sửa configs/data_paths.local.json hoặc set FASHION_EMBEDDING_CACHE."
    )

print("Core-7 folder :", CORE7_DIR)
print("Embedding cache:", CACHE_PATH)

## 4. Kiểm tra đủ 6 input JSONL

In [ ]:
SPLITS = ('train', 'valid', 'test')
POSITIVES = {
    split: CORE7_DIR / f'category_clean_{split}.jsonl'
    for split in SPLITS
}
METADATA = {
    split: CORE7_DIR / f'core7_item_metadata_v1_{split}.jsonl'
    for split in SPLITS
}
REPORT_PATH = CORE7_DIR / 'core7_embedding_validation_report.json'

for split in SPLITS:
    for kind, path in [('positive', POSITIVES[split]), ('metadata', METADATA[split])]:
        print(split, kind, path.name, 'exists=', path.exists())
        if not path.exists():
            raise FileNotFoundError(path)

## 5. Chạy cache audit + coverage join

Phần này chỉ đọc dữ liệu và ghi một report JSON nhỏ; không thay đổi ba dataset hiện tại.

In [ ]:
from src.data.validate_core7_embeddings import validate_core7_embedding_coverage

report = validate_core7_embedding_coverage(
    cache_path=CACHE_PATH,
    positives_by_split=POSITIVES,
    metadata_by_split=METADATA,
    report_path=REPORT_PATH,
)

print('Saved report:', REPORT_PATH)

## 6. Đọc kết quả ngắn gọn

In [ ]:
cache_report = report['cache']
print('CACHE')
print('  pass            :', cache_report['pass'])
print('  model           :', cache_report['model_id'])
print('  shape           :', (cache_report['embedding_row_count'], cache_report['embedding_dim']))
print('  usable items    :', cache_report['usable_item_count'])
print('  non-finite rows :', cache_report['nonfinite_row_count'])
print('  zero-norm rows  :', cache_report['zero_norm_row_count'])
print('  bad-norm rows   :', cache_report['bad_norm_row_count'])
print()

for split in SPLITS:
    split_report = report['splits'][split]
    print(split.upper())
    print('  pass                 :', split_report['pass'])
    print('  positive outfits     :', split_report['positive_sample_count'])
    print('  required unique items:', split_report['unique_required_item_count'])
    print('  embedding coverage   :', f"{split_report['embedding_coverage']:.4%}")
    print('  missing/invalid      :', split_report['missing_or_invalid_embedding_count'])
    print()

print('OVERALL PASS                 :', report['pass'])
print('REUSE CATEGORY-CLEAN AS FINAL:', report['reuse_category_clean_as_final'])
print('READY FOR NEGATIVE SAMPLING  :', report['ready_for_negative_sampling'])

## 7. Cách hiểu output

Nếu ba dòng cuối đều là `True`:

- không tạo thêm `final_clean_*.jsonl`;
- dùng chính `category_clean_train/valid/test.jsonl` làm final clean positives;
- file `core7_embedding_validation_report.json` là bằng chứng validation;
- bước tiếp theo là viết negative sampler mới.

Nếu `False`, xem các field `*_examples` trong report. Core code đã có hàm `repair_split`, nhưng **không tự sửa âm thầm** trước khi nhóm đọc nguyên nhân lỗi.